# Project FORESIGHT — Phase 8: ML Demand Forecasting

**Input:** Phase 6 `forecast_features.parquet`  
**Benchmarks (Phase 7 TEST):** SYNTHETIC Naive WAPE=72.8181 | UCI MA-30 WAPE=86.3870  

Models: RandomForest, HistGradientBoosting, LightGBM, XGBoost  
Selection on VALIDATION only; final scores on untouched TEST.


## 1–3. Load Features & Validation

In [1]:
import os, sys
import pandas as pd

BASE_DIR = os.path.abspath(".")
if os.path.basename(BASE_DIR) == "notebooks":
    BASE_DIR = os.path.abspath("..")
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from src.ml_forecasting import (
    load_feature_dataset, EXCLUDED_FIELDS, BASELINE_BENCHMARKS, ML_DIR, MODELS_DIR,
    write_ml_report,
)
from src.validate_ml_forecasting import run_validation

df = load_feature_dataset()
print("shape", df.shape)
print("split", df.split.value_counts().to_dict())
print("baselines", BASELINE_BENCHMARKS)
print("excluded fields:", list(EXCLUDED_FIELDS))
assert len(df) == 1995496
assert set(df.split.unique()) == {"train","validation","test"}
print("Phase 6 input OK")


shape (1995496, 62)
split {'train': 1569604, 'test': 226718, 'validation': 199174}
baselines {'SYNTHETIC': {'model': 'naive', 'WAPE': 72.8181, 'MAE': 5.2717, 'RMSE': 10.4688, 'sMAPE': 45.175}, 'UCI': {'model': 'moving_average_30', 'WAPE': 86.387, 'MAE': 18.8542, 'RMSE': 72.0799, 'sMAPE': 84.3796}}
excluded fields: ['units_sold', 'revenue', 'transaction_count', 'unique_customers', 'date', 'source_dataset', 'entity_id', 'product_key', 'sku_id', 'entity_type', 'split', 'insufficient_history']
Phase 6 input OK


## 4–10. Load Saved Phase 8 Results (trained pipeline)

In [2]:
metrics = pd.read_parquet(os.path.join(ML_DIR, "ml_model_metrics.parquet"))
preds = pd.read_parquet(os.path.join(ML_DIR, "ml_predictions.parquet"))
importance = pd.read_parquet(os.path.join(ML_DIR, "feature_importance.parquet"))
err = pd.read_parquet(os.path.join(ML_DIR, "ml_error_analysis.parquet"))

print("=== VALIDATION metrics ===")
print(metrics[metrics.split=="validation"][["source_dataset","model","MAE","RMSE","sMAPE","WAPE","training_time"]]
      .sort_values(["source_dataset","WAPE"]).to_string(index=False))

print("\n=== TEST metrics (+ baseline comparison) ===")
print(metrics[metrics.split=="test"][["source_dataset","model","MAE","RMSE","sMAPE","WAPE",
      "baseline_WAPE","wape_improvement_pct","selected"]]
      .sort_values(["source_dataset","WAPE"]).to_string(index=False))


=== VALIDATION metrics ===
source_dataset                  model     MAE    RMSE    sMAPE    WAPE  training_time
     SYNTHETIC               lightgbm  3.1115  6.0244 126.2837 40.1044          7.631
     SYNTHETIC          random_forest  3.2346  6.3700 144.7665 41.6907          9.148
     SYNTHETIC hist_gradient_boosting  3.2370  6.2381 122.2162 41.7219         13.145
     SYNTHETIC                xgboost  3.4757  6.5580 120.3094 44.7993         11.799
           UCI               lightgbm 16.5035 55.8952  80.2343 76.9218          1.749
           UCI                xgboost 16.6620 59.6071  78.2229 77.6605          3.051
           UCI hist_gradient_boosting 16.6774 58.9465  79.3097 77.7322          5.004
           UCI          random_forest 18.3963 57.6516  84.4070 85.7442          4.076

=== TEST metrics (+ baseline comparison) ===
source_dataset                  model     MAE    RMSE    sMAPE    WAPE  baseline_WAPE  wape_improvement_pct selected
     SYNTHETIC               lightgb

## 11. Baseline Comparison

In [3]:
for src in ["UCI","SYNTHETIC"]:
    best = metrics[(metrics.source_dataset==src)&(metrics.split=="test")&(metrics.selected==True)].iloc[0]
    base = BASELINE_BENCHMARKS[src]
    beat = best.WAPE < base["WAPE"]
    print(f"{src}: best={best.model} WAPE={best.WAPE:.4f} vs baseline {base['model']} {base['WAPE']}")
    print(f"  improvement_pct={best.wape_improvement_pct:.4f} | beat_baseline={beat}")


UCI: best=lightgbm WAPE=79.4710 vs baseline moving_average_30 86.387
  improvement_pct=8.0058 | beat_baseline=True
SYNTHETIC: best=lightgbm WAPE=38.8923 vs baseline naive 72.8181
  improvement_pct=46.5898 | beat_baseline=True


## 12. Feature Importance

In [4]:
for src in ["UCI","SYNTHETIC"]:
    sub = importance[(importance.source_dataset==src)&(importance.importance_type=="native")].sort_values("rank").head(20)
    print(f"\n{src} top-20 native importance:")
    print(sub[["rank","feature","importance"]].to_string(index=False))



UCI top-20 native importance:
 rank            feature  importance
    1 average_unit_price       795.0
    2        price_lag_1       491.0
    3   units_sold_lag_1       270.0
    4    rolling_mean_30       219.0
    5    demand_change_7       209.0
    6     rolling_std_30       207.0
    7  units_sold_lag_14       195.0
    8     rolling_mean_7       170.0
    9        day_of_year       163.0
   10   units_sold_lag_3       145.0
   11        day_of_week       143.0
   12    demand_growth_7       142.0
   13  units_sold_lag_21       130.0
   14    rolling_mean_14       120.0
   15   units_sold_lag_7       110.0
   16      rolling_std_7       107.0
   17       day_of_month       105.0
   18     rolling_std_14        97.0
   19   units_sold_lag_2        91.0
   20       week_of_year        89.0

SYNTHETIC top-20 native importance:
 rank            feature  importance
    1    rolling_mean_30       416.0
    2       on_order_qty       348.0
    3    rolling_mean_14       296.0
    4  

## 13–14. Product / Store / Error Analysis

In [5]:
for src in ["UCI","SYNTHETIC"]:
    bp = pd.read_parquet(os.path.join(ML_DIR, f"ml_metrics_by_product_{src.lower()}.parquet"))
    print(f"\n{src} best products:")
    print(bp.sort_values("WAPE").head(3)[["entity_id","product_key","WAPE","MAE"]].to_string(index=False))
    print(f"{src} worst products:")
    print(bp.sort_values("WAPE", ascending=False).head(3)[["entity_id","product_key","WAPE","MAE"]].to_string(index=False))
    hv_path = os.path.join(ML_DIR, f"ml_high_value_{src.lower()}.parquet")
    if os.path.exists(hv_path):
        print(pd.read_parquet(hv_path).to_string(index=False))

be = pd.read_parquet(os.path.join(ML_DIR, "ml_metrics_by_entity_synthetic.parquet"))
print("\nSynthetic stores:")
print(be.sort_values("WAPE")[["entity_id","MAE","RMSE","WAPE"]].to_string(index=False))
print("\nError analysis:")
print(err.to_string(index=False))



UCI best products:
entity_id product_key   WAPE    MAE
   ONLINE  UCI_40046A 0.3199 0.0384
   ONLINE  UCI_47586A 2.4252 0.1455
   ONLINE   UCI_21458 3.1002 0.3720
UCI worst products:
entity_id product_key       WAPE      MAE
   ONLINE  UCI_79063D 17554.5116 234.0602
   ONLINE  UCI_90214U  4455.5579 534.6669
   ONLINE   UCI_90071  2821.8625  34.4894
source_dataset       segment  n_skus  revenue_share_pct    model     MAE    RMSE     WAPE
           UCI  high_revenue     671              80.02 lightgbm 22.7186 43.4074  74.9877
           UCI lower_revenue    2497              19.98 lightgbm 12.5598 19.4243 187.3979

SYNTHETIC best products:
entity_id   product_key    WAPE    MAE
STORE_005 SYN_SKU_00068 14.5643 5.1223
STORE_001 SYN_SKU_00017 15.6644 1.4769
STORE_006 SYN_SKU_00068 15.6769 4.6423
SYNTHETIC worst products:
entity_id   product_key     WAPE    MAE
STORE_002 SYN_SKU_00043 158.7420 3.7904
STORE_008 SYN_SKU_00085 156.2656 3.7206
STORE_009 SYN_SKU_00043 147.2255 3.5154
source_dat

## 15–17. Saved Models & Predictions

In [6]:
import joblib
for src in ["uci","synthetic"]:
    path = os.path.join(MODELS_DIR, f"{src}_best_model.joblib")
    obj = joblib.load(path)
    print(src, "->", obj["model_name"], "features", len(obj["feature_names"]))
print("predictions rows", len(preds), preds.source_dataset.value_counts().to_dict())
print(preds.head(3).to_string(index=False))


uci -> lightgbm features 34
synthetic -> lightgbm features 50
predictions rows 226545 {'SYNTHETIC': 147000, 'UCI': 79545}
      date source_dataset entity_id product_key  actual_units_sold  predicted_units_sold    model
2011-09-26            UCI    ONLINE   UCI_10135                3.0             10.749200 lightgbm
2011-09-26            UCI    ONLINE UCI_15056BL               27.0              9.128003 lightgbm
2011-09-26            UCI    ONLINE  UCI_15056N               15.0             13.419193 lightgbm


## 18. Validation

In [7]:
result = run_validation()
print("VALIDATION:", result.summary())
assert result.failed == 0
print("Phase 8 COMPLETE — STOP before Phase 9.")


PHASE 8 ML FORECASTING VALIDATION

[1] Prerequisites & Artifacts
  [+] Phase 6 features exist
  [+] Predictions exist
  [+] Metrics exist
  [+] Feature importance exists
  [+] Training metadata exists
  [+] UCI best model file exists -- C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\models\uci_best_model.joblib
  [+] SYNTHETIC best model file exists -- C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\models\synthetic_best_model.joblib

[2] Predictions Integrity
  [+] Prediction column 'date'
  [+] Prediction column 'source_dataset'
  [+] Prediction column 'entity_id'
  [+] Prediction column 'product_key'
  [+] Prediction column 'actual_units_sold'
  [+] Prediction column 'predicted_units_sold'
  [+] Prediction column 'model'
  [+] No duplicate prediction keys -- duplicates=0
  [+] Predictions are numeric
  [+] Actual target present & numeric
  [+] No infinite predictions -- inf=0
  [+] No null predictions
  [+] No negative p

  [+] No shared entity across sources
  [+] No shared product_key across sources

[4] Chronology / No Test Leakage (metadata)
  [+] UCI: train_end < validation_start -- 2011-07-13 < 2011-07-14
  [+] UCI: validation_end < test_start -- 2011-09-25 < 2011-09-26
  [+] UCI: prediction dates within test window -- 2011-09-26..2011-12-09 vs 2011-09-26..2011-12-09
  [+] UCI: feature list non-empty
  [+] UCI: target not in feature list
  [+] UCI: revenue not in feature list
  [+] SYNTHETIC: train_end < validation_start -- 2025-03-13 < 2025-03-14
  [+] SYNTHETIC: validation_end < test_start -- 2025-08-06 < 2025-08-07
  [+] SYNTHETIC: prediction dates within test window -- 2025-08-07..2025-12-31 vs 2025-08-07..2025-12-31
  [+] SYNTHETIC: feature list non-empty
  [+] SYNTHETIC: target not in feature list
  [+] SYNTHETIC: revenue not in feature list

[5] Metrics & Baseline Comparison
  [+] Metric column MAE
  [+] MAE finite on non-null
  [+] Metric column RMSE
  [+] RMSE finite on non-null
  [+] Met

  [+] UCI model smoke predict works
  [+] SYNTHETIC model artifact loads -- dict
  [+] SYNTHETIC model smoke predict works

VALIDATION RESULT: 57/57 PASS
VALIDATION: 57/57 PASS
Phase 8 COMPLETE — STOP before Phase 9.


C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## 19. Summary

In [8]:
print("Models evaluated: random_forest, hist_gradient_boosting, lightgbm, xgboost")
for src in ["UCI","SYNTHETIC"]:
    best = metrics[(metrics.source_dataset==src)&(metrics.split=="test")&(metrics.selected==True)].iloc[0]
    print(src, dict(best[["model","MAE","RMSE","sMAPE","WAPE","baseline_WAPE","wape_improvement_pct","training_time"]]))
print("Validation:", result.summary())


Models evaluated: random_forest, hist_gradient_boosting, lightgbm, xgboost
UCI {'model': 'lightgbm', 'MAE': np.float64(17.3447), 'RMSE': np.float64(70.8952), 'sMAPE': np.float64(82.8734), 'WAPE': np.float64(79.471), 'baseline_WAPE': np.float64(86.387), 'wape_improvement_pct': np.float64(8.0058), 'training_time': np.float64(1.749)}
SYNTHETIC {'model': 'lightgbm', 'MAE': np.float64(2.8156), 'RMSE': np.float64(5.1469), 'sMAPE': np.float64(113.6813), 'WAPE': np.float64(38.8923), 'baseline_WAPE': np.float64(72.8181), 'wape_improvement_pct': np.float64(46.5898), 'training_time': np.float64(7.631)}
Validation: 57/57 PASS
